# FINPLE Monthly Metrics One-Click Pipeline

Step 114-2D offline rolling metrics and review-only overlay Colab workflow. The notebook exposes five operator sections and delegates calculation, source adapter selection, raw daily normalization, rolling CAGR, and audit generation to the repository module. In a fresh Colab runtime, the public FINPLE repository is cloned automatically. If cloning is unavailable, the notebook falls back to an uploaded Step 114-2D execution package ZIP. No private GitHub token is required.

## 1. Check settings

Confirm CONFIG only. The bootstrap first looks for an existing repository checkout, then clones the public FINPLE repository into `/content/FINPLE`, and finally falls back to an uploaded execution package ZIP. Step 114-2D remains offline-only and supports fixture, manual upload CSV, and provider-shaped synthetic public-source fixture adapter modes.

In [ ]:
from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/vip930sw/FINPLE.git"
CLONE_ROOT = Path("/content/FINPLE")

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "metrics_pipeline").exists() and (candidate / "data" / "fixtures" / "monthly-metrics").exists():
            return candidate
    return None

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

REPO_ROOT = find_repo_root(Path.cwd())
bootstrap_source = "existing_checkout" if REPO_ROOT is not None else None
uploaded_package_sha256 = None

if REPO_ROOT is None and CLONE_ROOT.exists():
    REPO_ROOT = find_repo_root(CLONE_ROOT)
    if REPO_ROOT is not None:
        bootstrap_source = "existing_clone"

if REPO_ROOT is None:
    if CLONE_ROOT.exists():
        shutil.rmtree(CLONE_ROOT)
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_ROOT)],
            check=True,
        )
        REPO_ROOT = find_repo_root(CLONE_ROOT)
        bootstrap_source = "git_clone"
    except (OSError, subprocess.CalledProcessError):
        try:
            from google.colab import files
            print("Git clone failed. Upload the Step 114-2D execution package ZIP containing scripts/metrics_pipeline and data/fixtures/monthly-metrics.")
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("No execution package ZIP uploaded.")
            zip_name = next(iter(uploaded))
            uploaded_package_sha256 = sha256_file(zip_name)
            extract_root = Path("/content/finple_step114_2d_execution_package")
            extract_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_name) as archive:
                archive.extractall(extract_root)
            REPO_ROOT = find_repo_root(extract_root)
            bootstrap_source = "uploaded_execution_package"
        except ImportError as exc:
            raise RuntimeError("Repository files were not found and the Colab upload helper is unavailable.") from exc

if REPO_ROOT is None:
    raise RuntimeError("FINPLE repository bootstrap failed.")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

repo_sha = None
git_dir = REPO_ROOT / ".git"
if git_dir.exists():
    try:
        repo_sha = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=REPO_ROOT,
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        repo_sha = None

RUN_CANDIDATE_MODE = False  # must be changed explicitly by the operator

CONFIG = {
    "metric_base_date": "2026-06-30",
    "market_scope": ["US", "KR"],
    "selected_cagr_policy": "rolling_median_all_markets",
    "current_price_display": False,
    "total_return_cagr_mode": "reference_only",
    "output_version": "2026_06",
    "input_mode": "fixture",  # fixture | manual_upload | public_source_fixture
    "input_dir": str(REPO_ROOT / "data" / "fixtures" / "monthly-metrics"),
    "output_dir": str(REPO_ROOT / "outputs" / "monthly-metrics" / "2026_06"),
    "deterministic_fixture": True,
    "random_seed": 1142,
}

CANDIDATE_CONFIG = {
    "metric_base_date": "2026-06-30",
    "market_scope": ["KR"],
    "output_version": "2026_06_candidate",
    "input_mode": "manual_upload_candidate",
    "input_dir": str(REPO_ROOT / "operator_uploads" / "candidate-package"),
    "output_dir": str(REPO_ROOT / "outputs" / "candidate-package" / "2026_06"),
}

print(f"Repository root: {REPO_ROOT}")
print(f"Bootstrap source: {bootstrap_source}")
if repo_sha:
    print(f"Repository SHA: {repo_sha}")
elif uploaded_package_sha256:
    print(f"Execution package SHA-256: {uploaded_package_sha256}")
else:
    print("Repository SHA: unavailable")
CONFIG

## 2. Check inputs

Fixture mode uses committed offline files only. Candidate mode is disabled by default and must be enabled explicitly with `RUN_CANDIDATE_MODE=True`. Candidate mode requires offline operator-uploaded files only and never calls external providers.


In [ ]:
active_config = CANDIDATE_CONFIG if RUN_CANDIDATE_MODE else CONFIG
input_dir = Path(active_config["input_dir"])
if RUN_CANDIDATE_MODE:
    if active_config.get("input_mode") != "manual_upload_candidate":
        raise ValueError("Candidate mode requires input_mode=manual_upload_candidate")
    required_inputs = [
        active_config.get("candidate_asset_master_file", "candidate_asset_master.csv"),
        active_config.get("benchmark_map_file", "benchmark_map.csv"),
        active_config.get("raw_daily_price_file", "raw_daily_prices.csv"),
        active_config.get("source_declaration_file", "source_declaration.json"),
        active_config.get("operator_submission_manifest_file", "operator_submission_manifest.json"),
    ]
    print("Candidate mode selected explicitly. Review all required files before running.")
else:
    required_inputs = ["candidates.csv", "benchmark_map.csv", "monthly_prices.csv"]
    if active_config["input_mode"] == "fixture":
        required_inputs.append("raw_daily_prices.csv")
    elif active_config["input_mode"] == "manual_upload":
        required_inputs.append(active_config.get("manual_upload_file", "manual_upload_raw_daily_prices.csv"))
    elif active_config["input_mode"] == "public_source_fixture":
        required_inputs.append(active_config.get("public_source_fixture_file", "public_source_fixture_prices.csv"))
    else:
        raise ValueError(f"Unsupported offline input_mode: {active_config['input_mode']}")

for file_name in required_inputs:
    path = input_dir / file_name
    print(f"{file_name}: {'OK' if path.exists() else 'MISSING'}")


## 3. Run pipeline

Run the single repository entrypoint. Fixture mode remains the default. Candidate mode uses the explicit offline production-candidate entrypoint and still returns `productionPublishReady=false` and `appExportApproved=false`.


In [ ]:
from scripts.metrics_pipeline import run_finple_monthly_metrics_pipeline, run_finple_production_candidate_package

if RUN_CANDIDATE_MODE:
    RESULT = run_finple_production_candidate_package(CANDIDATE_CONFIG)
else:
    RESULT = run_finple_monthly_metrics_pipeline(CONFIG)


## 4. Review summary

Review counts, rolling window counts, adapter summary, manifest versions, review-only overlay paths, and generated paths before using any output.

In [ ]:
print("FINPLE Monthly Metrics Pipeline Complete")
print(f"Mode: {'candidate' if RUN_CANDIDATE_MODE else 'fixture'}")
print(f"Fixture package ready: {RESULT['fixturePackageReady']}")
print(f"Candidate package ready: {RESULT.get('candidatePackageReady', False)}")
print(f"Production publish ready: {RESULT['productionPublishReady']}")
print(f"App export approved: {RESULT['appExportApproved']}")
if RUN_CANDIDATE_MODE:
    print(f"Candidate package id: {RESULT.get('candidatePackageId', '')}")
    print(f"Candidate package hash: {RESULT.get('candidatePackageHash', '')}")
    print(f"Blocking issues: {RESULT.get('blockingIssueCount', 0)}")
    print(f"Warning issues: {RESULT.get('warningIssueCount', 0)}")
else:
    print(f"Base date: {RESULT['metricBaseDate']}")
    print(f"Pipeline version: {RESULT['pipelineVersion']}")
    print(f"Source adapter summary: {RESULT['outputs']['sourceAdapterSummaryJson']}")
    print(f"Source adapter checkpoint: {RESULT['outputs']['sourceAdapterCheckpointJson']}")
    print(f"US review-only overlay: {RESULT['outputs']['usReviewOverlayCsv']}")
    print(f"KR review-only overlay: {RESULT['outputs']['krReviewOverlayCsv']}")
    for key, value in RESULT["summary"].items():
        print(f"{key}: {value}")
print(f"Output ZIP: {RESULT['outputs'].get('zipPackage', '')}")


## 5. Download output ZIP

Download the ZIP after checking the audit report and manifest. This ZIP is a fixture output package, not production app approval.

In [ ]:
zip_path = Path(RESULT["outputs"]["zipPackage"])
print(zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Colab download helper is unavailable in this runtime. Open the ZIP path above.")
